In [ ]:
"""
Compare FEM analytical data (angle=0 vs Zeng parallel, angle=90 vs Brenner
perpendicular). Data files contain (delta/a, drag-coeff ratio = lambda).
"""
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 30,
    "axes.labelsize": 30,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "legend.fontsize": 30,
    "lines.linewidth": 2.0,
    "axes.linewidth": 1.0,
})

HERE = os.getcwd()
import pathlib
p = pathlib.Path(HERE)
ROOT = None
for _ in range(8):
    candidate = p / 'src' / 'wall_corrections.py'
    if candidate.exists():
        ROOT = str(p)
        break
    if p.parent == p:
        break
    p = p.parent
if ROOT is None:
    # Fallback: previous behaviour (keeps compatibility)
    ROOT = os.path.abspath(os.path.join(HERE, '..', 'particle_inference'))
# Ensure project root is on sys.path
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.wall_corrections import WallCorrections


class _W(WallCorrections):
    pass


W = _W()

d0 = np.loadtxt(os.path.join(HERE, "angle-0.txt"))
d90 = np.loadtxt(os.path.join(HERE, "angle-90.txt"))

delta_par, lam_par_fem = d0[:, 0], d0[:, 1]
delta_perp, lam_perp_fem = d90[:, 0], d90[:, 1]

# Smooth analytical curves
delta_grid = np.logspace(-2, 2, 400)
zeng = W.zeng_parallel(delta_grid)
brenner = W.brenner_perpendicular(delta_grid)

# Evaluated at the FEM points (for tabulated comparison)
zeng_at_par = W.zeng_parallel(delta_par)
brenner_at_perp = W.brenner_perpendicular(delta_perp)

fig, (ax_par, ax_perp) = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

ax_par.plot(delta_grid, zeng, '-', color='tab:blue', label='Zeng et al. (2009)')
ax_par.plot(delta_par, lam_par_fem, 'o', color='k', markersize=12,
            markerfacecolor='white', label=r'FOM ($\theta=0^\circ$)')
ax_par.set_xscale('log')
ax_par.set_xlim(1e-2, 1e2)
ax_par.set_xticks([1e-2, 1e-1, 1e0, 1e1, 1e2])
ax_par.set_ylim(1, 4)
ax_par.set_yticks([1, 2, 3, 4])
ax_par.set_xlabel(r'$\delta/a$')
ax_par.set_ylabel(r'$f_\parallel$')
ax_par.grid(True, alpha=0.3)
ax_par.legend()

ax_perp.loglog(delta_grid, brenner, '-', color='tab:red', label='Brenner (1961)')
ax_perp.loglog(delta_perp, lam_perp_fem, 's', color='k', markersize=12,
               markerfacecolor='white', label=r'FOM ($\theta=90^\circ$)')
ax_perp.set_xlim(1e-2, 1e2)
ax_perp.set_xticks([1e-2, 1e-1, 1e0, 1e1, 1e2])
ax_perp.set_ylim(1, 1e2)
ax_perp.set_yticks([1, 1e1, 1e2])
ax_perp.set_xlabel(r'$\delta/a$')
ax_perp.set_ylabel(r'$f_\perp$')
ax_perp.grid(True, alpha=0.3)
ax_perp.legend()

plt.savefig('wall_correction.pdf')
plt.show()

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

# Use Agg only when not running interactively (Jupyter)
interactive = False
try:
    from IPython import get_ipython
    if get_ipython() is not None:
        interactive = True
except Exception:
    interactive = False
if not interactive:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Ensure the project `src` package is importable when running as a script
import os
import sys
import pathlib

HERE_SCRIPT = Path(__file__).resolve().parent if '__file__' in globals() else Path(os.getcwd())
# Walk up to find the project root that contains src/main.py
p = pathlib.Path(HERE_SCRIPT)
PROJECT_ROOT = None
for _ in range(8):
    candidate = p / 'src' / 'main.py'
    if candidate.exists():
        PROJECT_ROOT = str(p)
        break
    if p.parent == p:
        break
    p = p.parent
if PROJECT_ROOT is None:
    PROJECT_ROOT = str(HERE_SCRIPT.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.main import InferenceProcedure


ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path(os.getcwd())
# Prefer data directory at the project root if present
candidate_roots = [Path(PROJECT_ROOT), ROOT]
DATA_ROOT = None
for r in candidate_roots:
    p = Path(r) / "data" / "creep-8pi"
    if p.exists():
        DATA_ROOT = p
        break
if DATA_ROOT is None:
    DATA_ROOT = ROOT / "data" / "creep-8pi"  # fallback; will raise if missing

OUT_DIR = ROOT / "plots" / "creep_8pi_delta_overlays"

FORCE = 8.0 * np.pi
ETA_S = 0.5
ETA_P = 0.9
LAMBDA = 0.1
T_MAX = 0.5

ANGLES = [0, 45, 90]
DELTAS = [0.1, 0.01, 0.005]
CREEP_LOG_YLIM = (1e-4, 1)
CREEP_LOG_YTICKS = [1e-4, 1e-3, 1e-2, 1e-1, 1]


def _delta_label(delta: float) -> str:
    return f"{delta:g}"


def _load_creep_file(angle: int, delta: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    path = DATA_ROOT / f"creep-{angle}" / f"delta-{_delta_label(delta)}-{angle}.out"
    data = np.loadtxt(path)
    t = data[:, 0]
    x = data[:, 1]
    y = data[:, 2] - data[0, 2]
    keep = t <= T_MAX
    return t[keep], x[keep], y[keep]


def _model_series(
    model: InferenceProcedure,
    angle: int,
    delta: float,
    t: np.ndarray,
) -> np.ndarray:
    model.theta = angle
    return model.model_viscoelastic(
        ETA_S,
        ETA_P,
        LAMBDA,
        t,
        FORCE,
        delta,
    )


def _configure_matplotlib() -> None:
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 30,
        "axes.labelsize": 30,
        "xtick.labelsize": 30,
        "ytick.labelsize": 30,
        "legend.fontsize": 30,
        "lines.linewidth": 2.0,
        "axes.linewidth": 1.0,
    })
    


def _plot_angle(
    ax: plt.Axes,
    model: InferenceProcedure,
    angle: int,
    show_ylabel: bool = True,
) -> float:
    colors = plt.cm.tab10.colors
    all_y = []

    for idx, delta in enumerate(DELTAS):
        color = colors[idx]
        t, x_obs, y_obs = _load_creep_file(angle, delta)
        obs = y_obs if angle == 90 else x_obs
        t_model = np.linspace(0.0, min(T_MAX, float(t.max())), 600)
        pred = _model_series(model, angle, delta, t_model)

        ax.scatter(
            t,
            obs,
            s=14,
            color=color,
            marker="o",
            edgecolors="none",
            alpha=0.9,
            zorder=3,
        )
        ax.plot(t_model, pred, color=color, lw=2.2, zorder=2)
        all_y.extend([obs, pred])

    ylabel = r"$y(t)$" if angle == 90 else r"$x(t)$"
    ax.set_xlabel(r"$t$")
    if show_ylabel:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel(ylabel)
    ax.set_title(rf"$\theta = {angle}^\circ$")
    # ax.set_yscale('log')

    ax.set_xlim(0.0, T_MAX)
    ax.set_xticks([0.0, 0.25, 0.5])
    ax.grid(alpha=0.3)

    y_max = max(float(np.nanmax(y)) for y in all_y)
    y_top = y_max if y_max > 0 else 1.0
    ax.set_ylim(0.0, y_top)
    ax.set_yticks(np.linspace(0.0, y_top, 3))

    delta_handles = [
        Line2D(
            [0],
            [0],
            color=colors[idx],
            marker="o",
            lw=2.2,
            markersize=5,
            label=rf"$\delta/a = {_delta_label(delta)}$",
        )
        for idx, delta in enumerate(DELTAS)
    ]
    ax.legend(handles=delta_handles, loc="best", framealpha=0.9)
    return y_top


def main() -> None:
    _configure_matplotlib()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    model = InferenceProcedure(
        force=FORCE,
        material_model="viscoelastic",
        boundary_model="bounded",
        theta=45,
        a=1.0,
        t_unload=0.2,
        sigma_noise_percent=None,
    )

    for angle in ANGLES:
        fig, ax = plt.subplots(figsize=(8.0, 6.0))
        _plot_angle(ax, model, angle)
        fig.tight_layout()
        stem = OUT_DIR / f"creep_{angle}_delta_overlay"
        fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
        fig.savefig(stem.with_suffix(".png"), bbox_inches="tight")
        plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(24.0, 6.0))
    for ax, angle in zip(axes, ANGLES):
        _plot_angle(ax, model, angle, show_ylabel=True)
    for ax in axes:
        ax.set_yscale("log")
        ax.set_ylim(*CREEP_LOG_YLIM)
        ax.set_yticks(CREEP_LOG_YTICKS)
        ax.set_yticklabels([r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$", r"$10^{-1}$", r"$10^{0}$"])
    fig.tight_layout(w_pad=2.0)
    stem = OUT_DIR / "creep_delta_overlay_all_angles"
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".png"), bbox_inches="tight")
#    fig.show()

if __name__ == "__main__":
    main()
